In [0]:
def test_first_last_txn(spark):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, row_number, to_date, date_format

    data = [
        ("C1", "2024-01-01", 100),
        ("C1", "2024-01-10", 300),
        ("C2", "2024-01-02", 400),
        ("C2", "2024-01-20", 500)
    ]

    columns = ["customer_id", "txn_date", "amount"]
    df = spark.createDataFrame(data, columns) \
              .withColumn("txn_date", to_date("txn_date"))

    df = df.withColumn("month", date_format("txn_date", "yyyy-MM"))

    window_first = Window.partitionBy("customer_id", "month").orderBy("txn_date")
    window_last = Window.partitionBy("customer_id", "month").orderBy(col("txn_date").desc())

    df = df.withColumn("rn_first", row_number().over(window_first)) \
           .withColumn("rn_last", row_number().over(window_last))

    result_df = df.filter((col("rn_first") == 1) | (col("rn_last") == 1))

    result = {(r["customer_id"], str(r["txn_date"])) for r in result_df.collect()}

    expected = {
        ("C1", "2024-01-01"),
        ("C1", "2024-01-10"),
        ("C2", "2024-01-02"),
        ("C2", "2024-01-20")
    }

    try:
        assert result == expected
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")
        print("Expected:", expected)
        print("Got:", result)

# call function        
test_first_last_txn(spark)